# 18. 索引、切片与筛选

<!-- module-learning-arc:start -->
> **NumPy 模块主线｜第 3 / 6 步：定位与筛选业务片段**
>
> **持续应用背景：** 为区域仓库建立补货预警矩阵：把门店、商品、库存和需求组织成数组，逐步完成定位、广播计算、排序和抽样复核。
>
> **承接上一阶段：** 数组基础（ndarray）  →  **本章任务：** 索引、切片与筛选  →  **下一步：** 形状、合并与拆分
>
> **大作业连接：** 本章练习将成为《连锁门店补货预警矩阵》的一部分，最终需要把数组建模、风险筛选、广播计算和抽样复核组合成一份可执行的补货清单。
<!-- module-learning-arc:end -->


## 本章场景

记账助手的数据越攒越多：几十行还能用列表慢慢看，上千行就要靠 **NumPy 数组** 批量计算。数组和列表最大的区别是：索引、切片、筛选、统计全部按“整块数据”操作，速度快、写法短。

本章用一份 3×4 的销售/支出矩阵讲清四件事：**二维索引**（怎么定位一个元素）、**切片**（怎么取一块）、**布尔筛选**（怎么按条件取）、**高级索引**（怎么按顺序取多个位置）。学完你就能用数组完成“按条件筛选 + 汇总”的完整流程。



## 本章目标

学完本章，你将能够：

- **理解**：理解 ndarray 的索引、切片与布尔筛选。
- **操作**：能对数组做单点/切片/掩码取数与筛选。
- **迁移**：能从经营数据数组中按条件取出子集用于分析。


## 18.1 核心概念

**背景引入**：数组的“维度”是初学者最容易绕晕的地方。一维数组像一列数，二维数组像一张表——**逗号前是行，逗号后是列**。记住这一个口诀，二维索引、切片、筛选都能套用。

- 二维索引使用行、列两个维度。
- 布尔掩码通常需要与被筛选的维度形状一致；同形状筛选二维数组后，结果通常会按条件返回一维数组。
- 普通切片通常返回视图，高级索引通常返回副本。



## 18.2 方法分类速查

先用这张表建立本章的方法地图；每一行后面都有对应的独立示例或练习。

| 类别 | 常用方法或写法 | 主要用途 | 需要特别注意 |
| --- | --- | --- | --- |
| 二维索引 | `np.array()`、`matrix[1, 2]` | 二维数组使用行索引和列索引定位单个元素。 | 忘记二维数组需要分别指定行和列 |
| 切片 | `np.arange()`、`.reshape()`、`matrix[:2, 1:3]` | 切片左闭右开，可以同时选择行和列。 | 切片视图被修改后影响原数组 |
| 布尔筛选 | `np.array()`、`values[values >= 10]` | 布尔数组可以筛选满足条件的元素。 | 布尔掩码长度不一致 |
| np.where() | `np.array()`、`np.where()` | where可以根据条件从两个值或数组中选择结果。 | 忘记二维数组需要分别指定行和列 |
| 高级索引 | `np.array()`、`values[[4, 1, 3]` | 使用整数索引数组可以按指定顺序提取多个位置。 | 切片视图被修改后影响原数组 |



## 18.3 示例 1：二维索引与切片

**背景引入**：数组的"维度"是初学者最容易绕晕的地方。一维数组像一列数，二维数组像一张表——逗号前是行、逗号后是列。查找特定月份、取某区域数据，全靠索引切片。

**讲解**：sales[0] 取第 1 行（行索引从 0 开始）；sales[:, -1] 的冒号表示"所有行"，-1 表示最后一列；sales[:2, 1:3] 取前两行、第 2-3 列。

- 逗号前选择行，逗号后选择列；
- 行、列都从 0 开始数，冒号表示"这一段都要"；
- 负索引从尾部倒数：-1 是最后一列、-2 是倒数第二列；
- **口诀**：想取哪个维度，就把下标注在逗号对应那一侧。


<!-- math-foundation:chapter-18 -->
### 数学推导｜布尔筛选是指示条件

> 阅读方法：先跟着步骤理解每个量怎样产生，再看最后的可计算形式；不需要脱离业务场景死记公式。

**第 1 步｜逐元素判断。** 把条件结果记为掩码

$$
m_i=\mathbf{1}(A_i>\tau)
$$

**第 2 步｜掩码决定保留谁。** 被选中的元素满足 $m_i=1$，选中数量为

$$
n_{selected}=\sum_{i=1}^{n}m_i
$$

**第 3 步｜还能得到筛选比例。** $n_{selected}/n$ 就是超过阈值的样本占比。

**把上面的关系收束为本章计算式：**

$$
A_{\text{selected}}=\{A_i\mid A_i>\tau\}
$$

**符号解释：** $\tau$ 是阈值，集合保留所有满足条件的元素。

**代码对应：** `mask = A > threshold` 生成条件，`A[mask]` 返回满足条件的值。

**使用边界：** 筛选会改变样本构成；阈值必须来自业务规则或明确的统计口径。


In [ ]:
import numpy as np

sales = np.array(
    [
        [120, 150, 180, 210],
        [98, 132, 145, 170],
        [110, 128, 160, 188],
    ]
)
print("第1行:", sales[0])
print("最后一列:", sales[:, -1])
print("前两行中间两列:\n", sales[:2, 1:3])


## 18.4 示例 2：布尔筛选

**背景引入**：要找出"销售额≥160"的月份、筛出达标记录，条件判断得写到数据里去。布尔筛选就是用一个 True/False 的"筛子"，只保留条件成立的位置。

**讲解**：sales >= 160 生成与数组同形状的 True/False 掩码；sales[mask] 只保留 True 位置的元素；对行合计再用布尔筛选，就是"只看高销售月份"。

- 条件表达式直接生成与数组同形状的布尔数组；
- 布尔数组放进方括号 sales[mask]，原样保留满足条件的元素；
- **口诀**：条件成掩码，掩码进方括号，留下的就是你要的。


In [ ]:
mask = sales >= 160
print(mask)
print("达标值:", sales[mask])
monthly_total = sales.sum(axis=0)
print("高销售月份:", monthly_total[monthly_total > 450])


## 18.5 示例 3：高级索引

**背景引入**：有时要按指定顺序挑几行——先看周二、再看周一、最后周四，顺序还不能乱。整数索引数组让你按自己想要的顺序取位置。

**讲解**：values[[4, 1, 3]] 用整数索引数组按指定顺序提取位置——顺序可以任意、可以重复。

- 索引数组可以按任意顺序提取多个位置；
- 注意：高级索引返回**副本**，修改它不影响原数组（与切片视图相反）；
- **口诀**：要顺序、要重复，就用索引数组来挑。


In [ ]:
selected_rows = sales[[2, 0]]
selected_values = sales[[0, 2], [1, 3]]
print(selected_rows)
print("指定位置:", selected_values)
copy_part = sales[:2].copy()
copy_part[0, 0] = 999
print("原数组未改变:", sales[0, 0])


## 18.6 核心操作独立示例

下面每个代码单元格只演示一个核心方法、函数或语法操作。请先阅读方法名称和任务说明，再单独运行当前单元格；示例尽量自带最小输入，不要求依赖前一个单元格留下的变量。



In [ ]:
# 二维索引
# 二维数组使用行索引和列索引定位单个元素。
import numpy as np

matrix = np.array([[10, 20, 30], [40, 50, 60]])
print(matrix[1, 2])


In [ ]:
# 切片
# 切片左闭右开，可以同时选择行和列。
import numpy as np

matrix = np.arange(12).reshape(3, 4)
print(matrix[:2, 1:3])


In [ ]:
# 布尔筛选
# 布尔数组可以筛选满足条件的元素。
import numpy as np

values = np.array([12, 5, 18, 3, 20])
print(values[values >= 10])


In [ ]:
# np.where()
# where可以根据条件从两个值或数组中选择结果。
import numpy as np

values = np.array([52, 78, 91])
labels = np.where(values >= 60, "及格", "不及格")
print(labels)


In [ ]:
# 高级索引
# 使用整数索引数组可以按指定顺序提取多个位置。
import numpy as np

values = np.array([10, 20, 30, 40, 50])
print(values[[4, 1, 3]])


**练一练 15.6**：创建数组 values = np.array([10, 20, 30, 40, 50])，完成三件事：取最后两个元素；筛选出大于 25 的值；把第 2 个元素改成 99 后打印。


In [ ]:
# 请在下方填写代码
import numpy as np

# TODO：请在下方完成 —— 练一练 15.6：创建数组 values = np.array([10, 20, 30, 40, 50])，完成三件事：


In [ ]:
import numpy as np

values = np.array([10, 20, 30, 40, 50])
print(values[-2:])
big_values = list(values[values > 25])
print(big_values)
values[1] = 99
print(values)


**输出解读**：`values[-2:]` 取最后两个元素得到 `[40 50]`（负索引从尾部倒数）；`values[values > 25]` 用布尔筛选得到 `[30 40 50]`；执行 `values[1] = 99` 后，第 2 个元素（下标 1）变成 99，原数组被**原地修改**。三种取法可对比：切片取"一段"、布尔筛选取"满足条件的一组"、索引赋值"改回去"——这也是视图与副本差异的起点。


## 18.7 独立迁移练习

先预测 shape，再修改一个数组或筛选条件，解释结果变化。

先在下面单元格完成自己的版本；需要参考时再回看紧邻的示例或参考实现。



In [ ]:
# TODO: 在此粘贴或改写最接近的示例。
# 记录：我改了什么？预期会发生什么？实际观察到什么？
change_note = "待填写"
expected_change = "待填写"
observed_change = "运行后填写"
print({"修改": change_note, "预期": expected_change, "观察": observed_change})


## 18.8 本章实训：axis与布尔筛选

这一组实验专门训练“观察一个结果 → 只改一个变量 → 解释变化”。先运行第一个代码单元格，再运行第二个。



In [ ]:
import numpy as np

matrix = np.arange(1, 13).reshape(3, 4)
print("原数组：\n", matrix)
print("每行合计：", matrix.sum(axis=1))
print("每列合计：", matrix.sum(axis=0))


### 18.8.1 第一个结果怎么读

`axis=1` 保留行，沿列方向计算；`axis=0` 保留列，沿行方向计算。先看 shape，再解释结果长度。

请记录：输入是什么、输出是什么、输出支持了哪一个结论。



In [ ]:
even = matrix[matrix % 2 == 0]
print("偶数：", even)
print("偶数数量：", even.size)
print("偶数平均值：", even.mean())


### 18.8.2 第二个结果怎么读

第二个实验不改原数组，而是用布尔条件筛选新数组。请思考：如果条件改成 `matrix > 8`，输出会怎样变化？

迁移任务：把一个输入值、一个字段或一个图表参数换成自己的例子，再用一句话解释变化。



## 18.9 错误恢复：数组形状不匹配怎么办

真实数据和真实代码都会出问题。本节先观察问题，再用一个明确的检查或修复步骤恢复运行。



In [ ]:
import numpy as np

matrix = np.arange(6).reshape(2, 3)
try:
    result = matrix + np.array([10, 20])
except ValueError as error:
    print("形状问题：", type(error).__name__)
    result = matrix + np.array([10, 20, 30])
print("修复后的结果：")
print(result)


### 18.9.1 错误恢复步骤

1. 先看错误类型、字段或数据形状。
2. 判断问题发生在输入、处理中间结果还是输出。
3. 修复后重新检查结果，而不是只让代码不报错。

先看两个数组的 shape，再判断能否广播。修复不是随意 reshape，而是让数据结构和业务含义一致。

迁移任务：把示例中的输入换成一组会触发问题的数据，并记录你的修复规则。



## 18.10 易错点提醒

- 忘记二维数组需要分别指定行和列
- 切片视图被修改后影响原数组
- 布尔掩码长度不一致



## 18.11 练习与作业

1. 创建4×5数组
2. 提取最后两行
3. 筛选所有偶数并计算平均值

提交前检查：代码可从上到下运行，关键中间结果可核对，结论注明计算口径。

## 18.12 练习路径

1. **跟练**：先运行示例，确认输出结构，再完成“创建4×5数组”。
2. **独立完成**：不复制示例代码，完成“提取最后两行”，并保留一个中间结果用于检查。
3. **迁移挑战**：尝试“筛选所有偶数并计算平均值”，用一两句话说明你修改了什么。

### 18.12.1 完成标准

- 代码从上到下运行不报错，关键变量类型和形状符合预期。
- 至少输出一个可核对的数值、表格或图形，并写明计算口径。
- 结论能够回答任务问题，同时说明一个限制或未验证的假设。

### 18.12.2 分级提示

- **提示 1**：先复用示例中的数据结构和变量命名。
- **提示 2**：把任务拆成“准备数据 → 计算 → 检查 → 表达”四步。
- **提示 3**：运行隐藏答案前，先用 type()、shape、head() 或断言定位问题。



In [ ]:
import numpy as np

# TODO: 提取最后两行
# TODO: 筛选所有偶数
# TODO：请在下方完成 —— 15.12 练习与作业 1. 创建4×5数组 2. 提取最后两行 3. 筛选所有偶数并计算平均值 提交前检查：代码可从上


In [ ]:
import numpy as np

matrix = np.arange(1, 21).reshape(4, 5)
last_rows = matrix[-2:]
even_values = matrix[matrix % 2 == 0]
print(last_rows)
print("偶数:", even_values)
print("偶数平均值:", even_values.mean())


## 18.13 小结

掌握数组索引、切片、布尔掩码和高级索引，准确提取需要的数据。

**迁移思考**：

1. 如果需要提取所有大于平均值的元素并保持原有位置关系，应该用什么方法？
2. 为什么修改切片视图会影响原数组？什么时候需要使用 copy()？




### 18.13.1 你已经掌握

- 访问一维和二维元素
- 使用切片提取区域
- 根据条件筛选
- 理解视图与副本




### 18.13.2 验收标准

- 输入、计算和输出单元格完整。
- 关键变量类型、形状或数值可核对。
- 结论引用输出证据，并注明适用范围。




### 18.13.3 需要注意

- 忘记二维数组需要分别指定行和列
- 切片视图被修改后影响原数组
- 布尔掩码长度不一致




### 18.13.4 完成检查

- [ ] 能够访问一维和二维元素
- [ ] 能够使用切片提取区域
- [ ] 能够根据条件筛选
- [ ] 能够理解视图与副本




### 18.13.5 排错顺序

1. 从上到下重新运行依赖单元格。
2. 检查变量类型、列名、形状和缺失值。
3. 缩小输入范围，定位产生错误的最小步骤。
4. 修复后重新运行完整流程。


